In [ ]:
from pathlib import Path

import cv2
import numpy as np
from PIL import Image, ImageFilter

In [ ]:
# ============================================================
# CONFIG
# ============================================================

INPUT_ROOT = Path("13-class_0630/train")
OUTPUT_ROOT = Path("13-class_0630/Pipeline-5/train")

IMAGE_EXTS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp"
}

# Filter selection
SELECTED_METHODS = [1]

METHOD_MAP = {
    1: "Blur",
    2: "Edge_Enhance",
    3: "Edge_Enhance_More",
    4: "Detail",
    5: "Sharpen",
    6: "HE",
    7: "FHE",
    8: "CLAHE",
    9: "UnsharpMask",
}

In [ ]:
# ============================================================
# MODULE: PIL FILTERS
# ============================================================

def blur(image):
    return image.filter(ImageFilter.BLUR)


def edge_enhance(image):
    return image.filter(ImageFilter.EDGE_ENHANCE)


def edge_enhance_more(image):
    return image.filter(ImageFilter.EDGE_ENHANCE_MORE)


def detail(image):
    return image.filter(ImageFilter.DETAIL)


def sharpen(image):
    return image.filter(ImageFilter.SHARPEN)


def unsharp_mask(image):
    return image.filter(
        ImageFilter.UnsharpMask(
            radius=2,
            percent=120,
            threshold=3
        )
    )

In [ ]:
# ============================================================
# MODULE: OPENCV FILTERS
# ============================================================

def histogram_equalization(image_bgr):

    yuv = cv2.cvtColor(
        image_bgr,
        cv2.COLOR_BGR2YUV
    )

    y, u, v = cv2.split(yuv)

    y = cv2.equalizeHist(y)

    return cv2.cvtColor(
        cv2.merge((y, u, v)),
        cv2.COLOR_YUV2BGR
    )


def fhe(image_bgr):

    yuv = cv2.cvtColor(
        image_bgr,
        cv2.COLOR_BGR2YUV
    )

    y, u, v = cv2.split(yuv)

    filter = cv2.createCLAHE(
        clipLimit=4.0,
        tileGridSize=(8, 8)
    )

    y = filter.apply(y)

    return cv2.cvtColor(
        cv2.merge((y, u, v)),
        cv2.COLOR_YUV2BGR
    )


def clahe(image_bgr):

    yuv = cv2.cvtColor(
        image_bgr,
        cv2.COLOR_BGR2YUV
    )

    y, u, v = cv2.split(yuv)

    filter = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8, 8)
    )

    y = filter.apply(y)

    return cv2.cvtColor(
        cv2.merge((y, u, v)),
        cv2.COLOR_YUV2BGR
    )

In [ ]:
# ============================================================
# MODULE: IMAGE CONVERSION
# ============================================================

def bgr_to_pil(image):

    return Image.fromarray(
        cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB
        )
    )


def pil_to_bgr(image):

    return cv2.cvtColor(
        np.array(image),
        cv2.COLOR_RGB2BGR
    )

In [ ]:
# ============================================================
# MODULE: PIPELINE
# ============================================================

def apply_pipeline(image_bgr):

    image = bgr_to_pil(image_bgr)

    for method in SELECTED_METHODS:

        if method == 1:
            image = blur(image)

        elif method == 2:
            image = edge_enhance(image)

        elif method == 3:
            image = edge_enhance_more(image)

        elif method == 4:
            image = detail(image)

        elif method == 5:
            image = sharpen(image)

        elif method == 6:
            image = bgr_to_pil(
                histogram_equalization(
                    pil_to_bgr(image)
                )
            )

        elif method == 7:
            image = bgr_to_pil(
                fhe(
                    pil_to_bgr(image)
                )
            )

        elif method == 8:
            image = bgr_to_pil(
                clahe(
                    pil_to_bgr(image)
                )
            )

        elif method == 9:
            image = unsharp_mask(image)

    return np.array(image)

In [ ]:
# ============================================================
# PROCESS IMAGES
# ============================================================

for input_path in INPUT_ROOT.rglob("*"):

    if input_path.suffix.lower() not in IMAGE_EXTS:
        continue

    # Read image
    image = cv2.imread(str(input_path))

    if image is None:
        print(f"Cannot read image: {input_path}")
        continue

    # Apply selected filters
    processed_image = apply_pipeline(image)

    # Keep original folder structure
    relative_path = input_path.relative_to(INPUT_ROOT)
    output_path = OUTPUT_ROOT / relative_path

    # Create output folder
    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    # Save processed image
    cv2.imwrite(
        str(output_path),
        processed_image
    )

print("Processing complete.")